In [ ]:
from matplotlib import pyplot as plt
import h5py
import numpy as np
import bacco

%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.chdir("/cosmos_storage/home/fgmaion/MTNG-resims/src")
import utils

In [ ]:
mtng = bacco.utils.load_MTNG(adr="/cosmos_storage/simulations/TNG_Family/MTNG/", snap=264)
mtng.fof['halo_pos'][:,0] = (mtng.fof['halo_pos'][:,0] - 125) % 500
mtng.sub['pos'][:,0] = (mtng.sub['pos'][:,0] - 125) % 500

basedir = "/cosmos_storage/simulations/TNG_Family/MTNG-DM-Gadget4/MTNG-L500-2160-A/"

resolution_level = 1

sigma8 = 0.8159 #CHECK ME
ns     = 0.9667 #CHECK ME
tau    = 0.0965 #CHECK ME
numpart = int(1080**3)

mtng_dm = bacco.Simulation(basedir=basedir, halo_file="groups_265/fof_subhalo_tab_265", sim_format='gadget4_hdf5', fixedPk=True, sigma8=sigma8,\
    tau=tau, ns=ns, numpart=numpart, use_orphans=False, use_ids=False)

In [ ]:
dm_halo_sel = np.loadtxt("/cosmos_storage/simulations/TNG_Family/MN5_resims/resims_info/dm_halo_sel_1pmbin.txt").astype(int)
hydro_halo_sel = np.loadtxt("/cosmos_storage/simulations/TNG_Family/MN5_resims/resims_info/hydro_halo_sel_1pmbin.txt").astype(int)

In [ ]:
diff = mtng.fof['halo_pos'][hydro_halo_sel] - mtng_dm.fof['halo_pos'][dm_halo_sel]

In [ ]:
name_list = ["LH_"+str(i) for i in range(30)] + ["fiducial"]
snap = 264

sigma8 = 0.8159 #CHECK ME
ns     = 0.9667 #CHECK ME
tau    = 0.0965 #CHECK ME

zoom = {}
for i in range(len(name_list)):
    base = "/cosmos_storage/simulations/TNG_Family/MN5_resims/"+name_list[i]+"/hydro_output/"
    zoom[name_list[i]] = bacco.Simulation(basedir=base, halo_file="groups_{:03d}/fof_subhalo_tab_{:03d}".format(snap,snap),\
                            sim_format='TNG500', fixedPk=True, use_orphans=False, tau=tau, ns=ns, sigma8=sigma8,\
                            tree_file="groups_{:03d}/subhalo_prog_{:03d}".format(snap,snap), use_ids=True, numpart=4320)

In [ ]:
pol = {}
for i in range(len(name_list)):
    pol[name_list[i]] = (zoom[name_list[i]].fof['halo_mfof_type'][:,2] + zoom[name_list[i]].fof['halo_mfof_type'][:,3]) / zoom[name_list[i]].fof['halo_mfof_type'][:,1]

In [ ]:
fig, ax = plt.subplots(6, 5, dpi=200, figsize=(30, 25))

for i in range(6):
    for j in range(5):
        ax[i,j].set_xscale('log')
        ax[i,j].set_yscale('log')

        ax[i,j].hist(1e10*zoom[name_list[5*i+j]].fof['halo_m200c'], color='C3', bins=np.logspace(9,15,30))
        ax[i,j].hist(1e10*zoom[name_list[5*i+j]].fof['halo_m200c'][pol[name_list[5*i+j]]<0.002], color='C0', bins=np.logspace(9,15,30))

        ax[i,j].set_title(name_list[5*i+j])

        if j==0:
           ax[i,j].set_ylabel(r"$N_\mathrm{halos}$")
        if i==5:
            ax[i,j].set_xlabel(r"$M_\mathrm{200c}\ [M_\odot/h]$")

In [ ]:
fig, ax = plt.subplots(dpi=200, figsize=(5.5, 5))

ax.set_xscale('log')
ax.set_yscale('log')

ax.hist(1e10*zoom['fiducial'].fof['halo_m500c'], color='C3', bins=np.logspace(9,15,30))
ax.hist(1e10*zoom['fiducial'].fof['halo_m500c'][pol['fiducial']<0.002], color='C0', bins=np.logspace(9,15,30))

ax.set_title('fiducial')

ax.set_ylabel(r"$N_\mathrm{halos}$")
ax.set_xlabel(r"$M_\mathrm{200c}\ [M_\odot/h]$")

In [ ]:
xmatch = {}

for i in range(len(name_list)):
    xmatch[name_list[i]] = utils.cross_match(zoom[name_list[i]], snap=264)

In [ ]:
base = "/cosmos_storage/simulations/TNG_Family/MN5_resims/DM_only/output/"
zoom_dmo = bacco.Simulation(basedir=base, halo_file="groups_{:03d}/fof_subhalo_tab_{:03d}".format(snap,snap),\
                        sim_format='TNG500', fixedPk=True, use_orphans=False, tau=tau, ns=ns, sigma8=sigma8,\
                        tree_file="groups_{:03d}/subhalo_prog_{:03d}".format(snap,snap), use_ids=True, numpart=4320)

In [ ]:
xmatch_dmo = utils.cross_match(zoom_dmo, snap=264)

In [ ]:
xmatch_MTNG_dm = utils.cross_match(mtng_dm, snap=264)

In [ ]:
fig, ax = plt.subplots(dpi=100, figsize=(5.5, 5))

ax.set_yscale('log')
ax.set_xscale('log')

ax.set_title('Distance to Cross-Matched Haloes')

for i in range(len(name_list)-1):
    ax.hist(xmatch[name_list[i]]['d'], bins=np.logspace(-2.5,0.8,20), alpha=1, histtype='step', color='gray')

ax.hist(xmatch['fiducial']['d'], bins=np.logspace(-2.5,0.8,20), color='C3', alpha=0.5, label='fiducial')
ax.hist(xmatch_dmo['d'], bins=np.logspace(-2.5,0.8,20), alpha=0.5, label='Zoom-DMO', color='C0')
ax.hist(xmatch_MTNG_dm['d'], bins=np.logspace(-2.5,0.8,20), alpha=0.5, label='Full-DMO', color='C2', linestyle='--')

ax.legend()

ax.set_xlabel(r"$d\ [\mathrm{Mpc}/h]$")
ax.set_ylabel(r"$N_\mathrm{halos}$")

In [ ]:
def fgas_m500c(hydro):
    cont = hydro.fof['halo_masstype']

    m500c = hydro.fof['halo_m500c'] * 1e10
    r500c = hydro.fof['halo_r500c']
    
    sel = np.where(m500c > 1e10)[0]
    fgas = np.zeros(len(sel))
    for i in range(len(sel)):
        dist = np.sqrt(np.sum((hydro.sub['pos'] - hydro.fof['halo_pos'][sel[i]])**2, axis=1))
        mask = dist < r500c[sel[i]]
        mgas_500c = np.sum(hydro.sub['MassType'][mask,0]*1e10)
        fgas[i] = mgas_500c / m500c[sel[i]]

    return fgas

In [ ]:
old_hist = {}
new_hist = {}

for snap in range(100,245):
    LH0 = {}

    base = "/cosmos_storage/simulations/TNG_Family/MN5_resims/old_LH0/hydro_output/"
    LH0["old"] = bacco.Simulation(basedir=base, halo_file="groups_{:03d}/fof_subhalo_tab_{:03d}".format(snap,snap),\
                            sim_format='TNG500', fixedPk=True, use_orphans=False,\
                            tau=tau, ns=ns, sigma8=sigma8, tree_file="groups_{:03d}/subhalo_prog_{:03d}".format(snap,snap),\
                            use_ids=True, numpart=4320)

    base = "/cosmos_storage/simulations/TNG_Family/MN5_resims/LH_0/hydro_output/"
    LH0["new"] = bacco.Simulation(basedir=base, halo_file="groups_{:03d}/fof_subhalo_tab_{:03d}".format(snap,snap),\
                            sim_format='TNG500', fixedPk=True, use_orphans=False,\
                            tau=tau, ns=ns, sigma8=sigma8, tree_file="groups_{:03d}/subhalo_prog_{:03d}".format(snap,snap),\
                            use_ids=True, numpart=4320)

    # sel_old = LH0['old'].sub['MassType'][:,4] > 0.1
    # sel_new = LH0['new'].sub['MassType'][:,4] > 0.1

    old_hist[snap] = np.histogram(np.log10(LH0['old'].sub['MassType'][:,4]*1e10), bins=np.linspace(9, 12, 31))[0]
    new_hist[snap] = np.histogram(np.log10(LH0['new'].sub['MassType'][:,4]*1e10), bins=np.linspace(9, 12, 31))[0]

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(12,5))

ax[0].set_yscale('log')

for snap in range(100,245):
    ax[0].plot(bin_centers, old_hist[snap], alpha=1, color='C0')
    ax[0].plot(bin_centers, new_hist[snap], alpha=1, color='C3')

ratio = np.zeros((145, 30))
for snap in range(100,245):
    ratio[snap-100,:] = old_hist[snap]/new_hist[snap]
    ax[1].plot(bin_centers, old_hist[snap]/new_hist[snap], alpha=0.5, color='C0')

ax[1].plot(bin_centers, np.mean(ratio, axis=0), color='k', lw=2)

ax[1].axhline(1, color='k', ls='--')
ax[1].set_ylim(0.5, 1.5)
